# SuccessApp — Phase 3: Multimodal (photo → journal entry)

Use case: the user snaps a photo (handwritten journal page, a mood drawing, a screenshot of a bad email, a photo of their workout, etc.) and the app converts it into a structured journal entry — adjusting the goal graph if relevant.

**Why this matters for judging:** Gemma 4 is multimodal. Most submissions won't use this. Multimodal + on-device = strong differentiator.

## Step 1 — Reload model with the image-capable head

Phase 1/2 used `AutoModelForCausalLM` (text-only). For multimodal we need `AutoModelForImageTextToText`. Either restart the runtime first, or simply re-instantiate.

## Step 0 — Install (fresh runtime needs this)

Run this once per fresh Colab runtime. `transformers>=5.5.0` is required for the Gemma 4 architecture.

In [ ]:
!pip install -q -U "transformers>=5.5.0" accelerate
!pip install -q -U kagglehub

In [ ]:
import torch, gc
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from transformers import AutoProcessor, AutoModelForImageTextToText
import kagglehub
MODEL_PATH = kagglehub.model_download('google/gemma-4/transformers/gemma-4-e2b-it')
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto').eval()
print(f'Loaded on {model.device} | VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Step 2 — Image input helper

In [ ]:
from PIL import Image
import io, base64, json, re
from google.colab import files

def load_image(path_or_url):
    if path_or_url.startswith('http'):
        import urllib.request
        return Image.open(urllib.request.urlopen(path_or_url)).convert('RGB')
    return Image.open(path_or_url).convert('RGB')

def upload_image_interactive():
    up = files.upload()
    fname = next(iter(up))
    return Image.open(io.BytesIO(up[fname])).convert('RGB')

## Step 3 — Journal-from-photo system prompt

In [ ]:
PHOTO_JOURNAL_SYSTEM = '''You are SuccessApp's photo journal helper.
The user shared an image alongside (optional) text. Produce a structured journal entry capturing what is meaningfully in the photo and how it connects to their wellbeing or goals.

## What to look for
- Handwritten notes -> transcribe key lines (max 6) into key_themes
- Screenshots of messages/emails -> capture the emotional tone, not the content verbatim
- Drawings or doodles -> describe mood/themes you see
- Photos of workouts, meals, places -> infer the activity and its emotional flavor
- Selfies -> infer mood from expression cautiously; never claim certainty

## Safety
- Never invent text you cannot actually read.
- Never identify people by name from a photo.
- If the image shows self-harm, weapons, or a person in distress in a clearly dangerous way, set crisis_flag=true and leave other fields minimal.

## Output STRICT JSON ONLY (no markdown):
{
  "summary": "<2-3 sentences about what is in the image and how it relates to wellbeing>",
  "detected_text": "<verbatim text you could read in the image, or empty string>",
  "mood_score": 1-10,
  "key_themes": ["<short noun phrases>"],
  "connected_goal_hint": "<short noun phrase OR null>",
  "crisis_flag": true|false
}'''

## Step 4 — Multimodal generate

In [ ]:
def extract_json(raw: str):
    raw = raw.strip()
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    candidate = m.group(1) if m else None
    if not candidate:
        s, e = raw.find('{'), raw.rfind('}')
        if s != -1 and e > s:
            candidate = raw[s:e+1]
    if not candidate: return None
    try: return json.loads(candidate)
    except json.JSONDecodeError: return None

def photo_journal(image, user_caption: str = ''):
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': PHOTO_JOURNAL_SYSTEM}]},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': user_caption or 'Here is the image I want to journal about.'},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    input_len = inputs['input_ids'].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    raw = processor.decode(out[0][input_len:], skip_special_tokens=True)
    return {'raw': raw, 'parsed': extract_json(raw)}

## Step 5 — Try it on a sample image

Option A — upload one of your own photos:

In [ ]:
# img = upload_image_interactive()
# r = photo_journal(img, user_caption='Snapshot of my morning run today, felt good for the first time in weeks.')
# print(r['raw']); print('\nPARSED:', json.dumps(r['parsed'], indent=2))

Option B — a public placeholder image (so the cell runs without manual upload):

In [ ]:
img = load_image('https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800')  # mountain landscape
r = photo_journal(img, user_caption='Went hiking solo this weekend to clear my head.')
print(r['raw'][:800])
print('\nPARSED:', json.dumps(r['parsed'], indent=2) if r['parsed'] else 'JSON PARSE FAILED')

## Step 6 — Plug the photo entry into the agentic loop

After producing a photo_journal JSON, we can call `save_journal_entry` with mapped fields and (if `connected_goal_hint`) potentially update an existing goal graph. The Flutter app will do this natively; in the notebook you can append to `STATE['journal_entries']` from Phase 2.

In [ ]:
from datetime import date
def photo_to_journal_entry(parsed: dict):
    return {
        'date': date.today().isoformat(),
        'mood_score': parsed.get('mood_score', 5),
        'key_themes': parsed.get('key_themes', []),
        'wins': [],
        'concerns': [],
        'goal_progress_notes': parsed.get('connected_goal_hint') or '',
        'reflection_prompt_for_tomorrow': 'What did today\'s image remind you of?',
        'source': 'photo',
        'summary': parsed.get('summary', ''),
    }

if r['parsed']:
    entry = photo_to_journal_entry(r['parsed'])
    print(json.dumps(entry, indent=2))

## Phase 3 Definition of Done
- Photo journaling returns valid JSON for landscape, handwritten note, and screenshot-of-message inputs
- crisis_flag triggers correctly on visibly distressing images (self-harm imagery)
- Does NOT identify people by name from photos